# UNSW-NB15 — відтворюваний експеримент IDS

**Мета:** повторити офіційний протокол магістерської роботи й отримати незалежні метрики. Notebook **не містить вигаданих результатів або готових логів**. Потрібні `data/UNSW_NB15_training-set.csv`, `data/UNSW_NB15_testing-set.csv` та інстальовані залежності. За першого виконання відбудеться повне навчання двох моделей.

In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
from IPython.display import display, Markdown, Image

HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name == 'notebooks' else HERE
TRAIN = ROOT / 'data' / 'UNSW_NB15_training-set.csv'
TEST = ROOT / 'data' / 'UNSW_NB15_testing-set.csv'
assert TRAIN.exists(), f'Відсутній {TRAIN}'
assert TEST.exists(), f'Відсутній {TEST}'
print('Project:', ROOT)
print('Train:', TRAIN.name, 'Test:', TEST.name)

## Реальний запуск

Нижче викликається `scripts/run_official_experiment.py`, який використовує класи `ModelTrainer` та `PredictionService`. Параметри та тестова вибірка не підганяються під диплом. Повторний запуск потребує іншого `--out` для збереження попередніх доказів.

In [ ]:
REPORT = ROOT / 'documentation' / 'experiments' / 'official_unsw_nb15'
cmd = [sys.executable, str(ROOT / 'scripts' / 'run_official_experiment.py'),
       '--train-file', str(TRAIN), '--test-file', str(TEST), '--out', str(REPORT), '--seed', '42']
subprocess.run(cmd, cwd=ROOT, check=True)
print('Звіт:', REPORT)

## Фактичні результати

Візуалізація зчитує **створені програмою** артефакти. Якщо експеримент ще не виконаний, файлів не існує.

In [ ]:
metrics = json.loads((REPORT / 'test' / 'metrics.json').read_text(encoding='utf-8'))
compare = pd.read_csv(REPORT / 'train' / 'validation_comparison.csv')
print('Валідація:')
display(compare)
print('Незалежний тест:')
display(pd.DataFrame([metrics])[['rows','TN','FP','FN','TP','precision','recall','f1','accuracy','fpr']])
display(Image(filename=str(REPORT / 'test' / 'confusion_matrix.png')))

In [ ]:
print((REPORT / 'SUMMARY.md').read_text(encoding='utf-8'))
print('Машинні логи:')
for logfile in (REPORT / 'train' / 'events.jsonl', REPORT / 'test' / 'events.jsonl'):
    print('---', logfile)
    print(logfile.read_text(encoding='utf-8'))